In [74]:
import pytreenet as ptn
from qutip import coherent
from copy import deepcopy
import numpy as np
np.random.seed(345)

In [75]:

#   I appplied a different approach in order to use built-in functions as much as possible. 
# For now, I will neglect bond_expansion and T3NS optimization, to focus on the issue.
# Morover, I would not normalize states, and just consider the normalization factor in the 
ptn.expectation_value_Lindblad 
# to just compare with exact results before going further in time. 

# So, first of all, I initiated tdvp with vecrotized_pho with identifier "Site(i,j)" for 
# |ket> part and "Node(i,j)" for <bra| part and corresponding effective Hamiltonian TTNO
# Then after each sweep, I devectorized the vecrotized_pho with
ptn.devectorize_pho
# into two TTN;i.e. <bra| and |ket> parts with identifier named "Vertex(i,j)". 
# I only added one attribute "self.connections" for this step.

# Then, I had to deal with the extra virtual bond that connects "Node(0,0)" to "Site(0,0)".
# So, I QR-decomposed the "Vertex(0,0)" with 
ptn.split_root_qr
# and kept the Q for "Vertex(0,0)" and pushed the R to the new child named "Vertex(0,0)_R".
# and insert Identity to the corresponding TTNO node in order to calculate the expectation value.

# So, for calculating the expectation value using the finction
ptn.expectation_value_Lindblad
# which contracts TTNO locally to |ket> with
ptn.contract_ttno_with_ttn 
# and then contracts it with <bra| with built-in function
ptn.contract_two_ttns 

# QUESTION: Is contract_ttno_with_ttn good for this step or its more efficient to change
ptn.expectation_value
# for calculating <bra|TTNO|ket> ?

# Additionally, I useded to functions :
ptn.adjust_bra_to_ket
# and
ptn.adjust_ttn1_structure_to_ttn2
# to preserve the initial structure of TTNs due to chnage after 
# applying canical_form and run_one_time_step functions.

# Initialize state with product of coherent states |$\alpha$>


In [76]:
def product_state(ttn, bond_dim=2 , physical_dim= 2):
    product_state = deepcopy(ttn)
    A = np.array([1,1])
    alpha = 1
    #A = np.array(coherent(physical_dim , alpha).full())
    for node_id in product_state.nodes.keys():
        n = product_state.tensors[node_id].ndim - 1
        tensor = A.reshape((1,) * n + (physical_dim,))
        T = np.pad(tensor, n*((0, bond_dim-1),) + ((0, 0),))
        product_state.tensors[node_id] = T
        product_state.nodes[node_id].link_tensor(T)  
    return product_state

# Initialize vectorized_pho

In [77]:
# local physical dimension
d = 2

shapes = {
    (0, 0): (3, 5, 6, d),
    (0, 1): (3, 7, d),
    (0, 2): (7, 8, d),
    (1, 0): (5, 5, d),
    (1, 1): (9, d),
    (1, 2): (8, d),
    (2, 0): (5, 6, d),
    (2, 1): (6, 9, 3, d),
    (2, 2): (3, d)
}


sites = {
    (i, j): ptn.random_tensor_node(shapes[(i, j)], identifier=f"Site({i},{j})") for i in range(3) for j in range(3)
}

vectorized_pho = ptn.TreeTensorNetworkState()

vectorized_pho.add_root(sites[(0, 0)][0], sites[(0, 0)][1])

connections = [
    ((0, 0), (0, 1), 0, 0),
    ((0, 1), (0, 2), 1, 0),
    ((0, 2), (1, 2), 1, 0),
    ((0, 0), (1, 0), 1, 0),
    ((1, 0), (2, 0), 1, 0),
    ((2, 0), (2, 1), 1, 0),
    ((2, 1), (1, 1), 1, 0),
    ((2, 1), (2, 2), 2, 0)]


for (parent, child, parent_leg, child_leg) in connections:
    parent_id = f"Site({parent[0]},{parent[1]})"
    child_id = f"Site({child[0]},{child[1]})"
    vectorized_pho.add_child_to_parent(sites[child][0], sites[child][1], child_leg, parent_id, parent_leg)

vectorized_pho = product_state(vectorized_pho , bond_dim= 5, physical_dim = d)

nodes = {
    (i, j): (ptn.Node(tensor=vectorized_pho.tensors[f"Site({i},{j})"] , identifier=f"Node({i},{j})"), vectorized_pho.tensors[f"Site({i},{j})"]) for i in range(3) for j in range(3)
}

vectorized_pho.add_child_to_parent(nodes[(0,0)][0], nodes[(0,0)][1], 2, "Site(0,0)", 2)

connections = [
    ((0, 0), (0, 1), 1, 0),
    ((0, 1), (0, 2), 1, 0),
    ((0, 2), (1, 2), 1, 0),
    ((0, 0), (1, 0), 2, 0),
    ((1, 0), (2, 0), 1, 0),
    ((2, 0), (2, 1), 1, 0),
    ((2, 1), (1, 1), 1, 0),
    ((2, 1), (2, 2), 2, 0),
]

for (parent, child, parent_leg, child_leg) in connections:
    parent_id = f"Node({parent[0]},{parent[1]})"
    vectorized_pho.add_child_to_parent(nodes[child][0], nodes[child][1], child_leg, parent_id, parent_leg)

# Effective Hamiltonian

In [78]:
def get_neighbors_Site(x, y, Lx, Ly):
  neighbors = []
  
  # Right neighbor
  if x < Lx - 1:
      neighbors.append(f"Site({x+1},{y})")
  
  # Up neighbor
  if y < Ly - 1:
      neighbors.append(f"Site({x},{y+1})")
  
  return neighbors

def get_neighbors_Node(x, y, Lx, Ly):
  neighbors = []

  # Right neighbor
  if x < Lx - 1:
      neighbors.append(f"Node({x+1},{y})")
  
  # Up neighbor
  if y < Ly - 1:
      neighbors.append(f"Node({x},{y+1})")
  
  return neighbors

def get_neighbors_Vertex(x, y, Lx, Ly):
  neighbors = []

  # Right neighbor
  if x < Lx - 1:
      neighbors.append(f"Vertex({x+1},{y})")
  
  # Up neighbor
  if y < Ly - 1:
      neighbors.append(f"Vertex({x},{y+1})")
  
  return neighbors

In [79]:
def Liouville(t, U, gamma, m, L, Lx, Ly, d):
    creation_op, annihilation_op, number_op = ptn.bosonic_operators(d)
    
    conversion_dict = {
        "b^dagger": creation_op,
        "b": annihilation_op,
        f"I{d}": np.eye(d)
    }
    
    conversion_dict.update({
        "it * b^dagger": t*1j * creation_op,
        "it * b": t*1j * annihilation_op,
        "-iU * n * (n - 1)": -U*1j * number_op @ (number_op - np.eye(d)),
        "im*n": m*1j*number_op
    })
    
    terms = []
    
    # Hopping terms
    for x in range(Lx):
        for y in range(Ly):
            current_site = f"Site({x},{y})"
            neighbors = get_neighbors_Site(x, y, Lx, Ly)            
            for neighbor in neighbors:
                terms.append(ptn.TensorProduct({current_site: "it * b^dagger", neighbor: "b"}))
                terms.append(ptn.TensorProduct({current_site: "it * b", neighbor: "b^dagger"}))
                

    
    # On-site interaction terms
    for x in range(Lx):
        for y in range(Ly):
            current_site = f"Site({x},{y})"
            terms.append(ptn.TensorProduct({current_site: "-iU * n * (n - 1)"}))

    # Chemical potential terms
    for x in range(Lx):
        for y in range(Ly):
            current_site = f"Site({x},{y})"
            terms.append(ptn.TensorProduct({current_site: "im*n"}))        
    
    H1 = ptn.Hamiltonian(terms, conversion_dict)
    
    conversion_dict = {
        "b^dagger.T": creation_op,
        "b.T": annihilation_op,
        f"I{d}": np.eye(d)
    }
    
    conversion_dict.update({
        "-it * b^dagger.T": -t*1j * creation_op,
        "-it * b.T": -t*1j * annihilation_op,
        "iU * n * (n - 1).T": (U*1j * number_op @ (number_op - np.eye(d))),
        "-im*n.T": -m*1j* number_op
    })
    
    terms = []
    
    # Hopping terms
    for x in range(Lx):
        for y in range(Ly):
            current_site = f"Node({x},{y})"
            neighbors = get_neighbors_Node(x, y, Lx, Ly)
            for neighbor in neighbors:
                terms.append(ptn.TensorProduct({current_site: "-it * b^dagger.T", neighbor: "b.T"}))
                terms.append(ptn.TensorProduct({current_site: "-it * b.T", neighbor: "b^dagger.T"}))

    # On-site interaction terms
    for x in range(Lx):
        for y in range(Ly):
            current_site = f"Node({x},{y})"
            terms.append(ptn.TensorProduct({current_site: "iU * n * (n - 1).T"}))    

    # Chemical potential terms
    for x in range(Lx):
        for y in range(Ly):
            current_site = f"Node({x},{y})"
            terms.append(ptn.TensorProduct({current_site: "-im*n.T"}))
            
    H2 = ptn.Hamiltonian(terms, conversion_dict)
    H1.__add__(H2)

        
    conversion_dict = {    
    "L": np.sqrt(gamma) * L,
    "L^dagger.T": np.sqrt(gamma) * L.conj(),
    "-1/2 (L^dagger @ L) " : -1/2 * gamma * L.conj().T @ L,
    "-1/2 (L^dagger @ L).T": -1/2 * gamma * (L.conj().T @ L).T}
    
    terms = []
    for x in range(Lx):
        for y in range(Ly):
            out_site = f"Node({x},{y})"
            in_site = f"Site({x},{y})"
            terms.append(ptn.TensorProduct({in_site: "L" , out_site: "L^dagger.T"}))
            terms.append(ptn.TensorProduct({in_site: "-1/2 (L^dagger @ L) "}))
            terms.append(ptn.TensorProduct({out_site: "-1/2 (L^dagger @ L).T"}))

    H3 = ptn.Hamiltonian(terms, conversion_dict)
    H1.__add__(H3)
    return H1

# Total Occupation 

In [80]:
def Number_op_total(Lx, Ly, dim=2):
    creation_op, annihilation_op, number_op = ptn.bosonic_operators(dim)
    conversion_dict = {"n": number_op , f"I{dim}": np.eye(dim)}
    for dim in range(2, 200):
        conversion_dict[f"I{dim}"] = np.eye(dim)

    terms = []
    for x in range(Lx):
        for y in range(Ly):
            current_site = f"Vertex({x},{y})"
            terms.append(ptn.TensorProduct({current_site: "n"}))

    return ptn.Hamiltonian(terms, conversion_dict) 

def BoseHubbard_ham(t, U, m, Lx, Ly, d):
    creation_op, annihilation_op, number_op = ptn.bosonic_operators(d)

    conversion_dict = {
        "b^dagger": creation_op,
        "b": annihilation_op,
        f"I{d}": np.eye(d)}
    
    conversion_dict.update({
        "-t * b^dagger": -t * creation_op,
        "-t * b": -t * annihilation_op,
        "U * n * (n - 1)": U * number_op @ (number_op - np.eye(d)),
        "-m*n": -m * number_op
    })

    for dim in range(2, 200):
        conversion_dict[f"I{dim}"] = np.eye(dim)    
    terms = []
    
    # Hopping terms
    for x in range(Lx):
        for y in range(Ly):
            current_site = f"Vertex({x},{y})"
            neighbors = get_neighbors_Vertex(x, y, Lx, Ly)
            for neighbor in neighbors:
                terms.append(ptn.TensorProduct({current_site: "-t * b^dagger", neighbor: "b"}))
                terms.append(ptn.TensorProduct({current_site: "-t * b", neighbor: "b^dagger"}))


    # On-site interaction terms
    for x in range(Lx):
        for y in range(Ly):
            current_site = f"Vertex({x},{y})"
            terms.append(ptn.TensorProduct({current_site: "U * n * (n - 1)"}))
    
    # Chemical potential terms
    for x in range(Lx):
        for y in range(Ly):
            current_site = f"Vertex({x},{y})"
            terms.append(ptn.TensorProduct({current_site: "-m*n"}))

    return ptn.Hamiltonian(terms, conversion_dict)



# Define SecondOrderOneSiteTDVP
gamma = 0

In [81]:
ket , bra = ptn.devectorize_pho(vectorized_pho , connections)

t = 0.4
U = 0.8
m = 0.4
creation_op, annihilation_op, number_op = ptn.bosonic_operators(d)
L = annihilation_op 
gamma = 0

# TTNO : Number operator
N = Number_op_total(3, 3, d)
N = N.pad_with_identities(ket, symbolic= True)
N = ptn.TTNO.from_hamiltonian(N, ket)

# TTNO : Bose-Hubbard Hamiltonian
H = BoseHubbard_ham(t, U, m, 3, 3, d)
H = H.pad_with_identities(ket, symbolic= True)
H = ptn.TTNO.from_hamiltonian(H, ket)

# TTNO : Identity operator
I_pho = ptn.TTNO.Identity(vectorized_pho)
I_ket = ptn.TTNO.Identity(ket)
# TTNO : Liouville operator 
H1 = Liouville(t, U, gamma, m ,L, 3, 3, d)
H1 = H1.pad_with_identities(vectorized_pho , symbolic= True)
L_fancy = ptn.TTNO.from_hamiltonian(H1, vectorized_pho)

connections = [ ((0, 0), (0, 1), 1, 0),
                ((0, 1), (0, 2), 1, 0),
                ((0, 2), (1, 2), 1, 0),
                ((0, 0), (1, 0), 2, 0),
                ((1, 0), (2, 0), 1, 0),
                ((2, 0), (2, 1), 1, 0),
                ((2, 1), (1, 1), 1, 0),
                ((2, 1), (2, 2), 2, 0)]

def transpose_last_two(tensor):
    # Ensure the last two dimensions are (2, 2)
    if tensor.shape[-2:] != (2, 2):
        raise ValueError("The last two dimensions must be (2, 2)")

    # Use swapaxes to swap the last two dimensions
    transposed_tensor = tensor.swapaxes(-1, -2)
    return transposed_tensor


In [82]:
c

In [83]:
tdvp_Lindblad.run_Lindblad(evaluation_time=2)
times = tdvp_Lindblad.times

# constant results for gamma = 0

In [38]:
tdvp_Lindblad.operator_results()[0]

In [72]:
import matplotlib.pyplot as plt

fig1, axs1 = plt.subplots(1, 1, sharex=True, figsize=(5, 5))

axs1.plot( tdvp_Lindblad.operator_results()[0] , label="gamma = 0")
#axs1.plot( np.imag(norm_2) , label="gamma = 0")

axs1.set_xlabel("Time $t$")
axs1.set_ylabel("Total Occupation ")
axs1.grid(True)
axs1.legend()

# Exact results 
gamma = 1

In [12]:
from qutip import *
import numpy as np

# Define parameters
t = 0.4  # Hopping strength
U = 0.8  # On-site interaction strength
m = 0.4  # Chemical potential
gamma_relax = 1  # Relaxation rate

# Reduced lattice dimensions
Nx = 3  # Number of sites along x-direction
Ny = 3  # Number of sites along y-direction
N = Nx * Ny  # Total number of sites

# Reduced maximum number of bosons per site
nmax = 1

# Precompute the operators for each site
a_list = []
adag_list = []
n_list = []
si = qeye(nmax + 1)  # Identity operator for a single site
for n in range(N):
    op_list = [si] * N
    op_list[n] = destroy(nmax + 1)
    a_op = tensor(op_list)
    a_list.append(a_op)
    adag_list.append(a_op.dag())
    n_list.append(a_op.dag() * a_op)


# Function to map 2D lattice coordinates (i, j) to a site index
def site(i, j):
    return i + j * Nx

# Initialize the Hamiltonian
H = 0

# Build the Hamiltonian by summing over sites
for i in range(Nx):
    for j in range(Ny):
        n = site(i, j)
        H += 0.5 * U * n_list[n] * (n_list[n] - 1) - m * n_list[n]
        if i < Nx - 1:
            n_right = site(i + 1, j)
            H += -t * (adag_list[n] * a_list[n_right] + adag_list[n_right] * a_list[n])
        if j < Ny - 1:
            n_up = site(i, j + 1)
            H += -t * (adag_list[n] * a_list[n_up] + adag_list[n_up] * a_list[n])

# Initial state: product state of maximum occupation

#psi0 = tensor([basis(nmax + 1, nmax) for _ in range(N)]).unit()
# psi0 = tensor([(basis(nmax + 1, 0) + basis(nmax + 1, 1)).unit() for _ in range(N)])
alpha = 1
psi0 = tensor([coherent(nmax + 1, alpha) for _ in range(N)])
# Reduced simulation time and increased time step
total_time = 4  # Total time in seconds
time_step = 0.02  # Time step in seconds
tlist = np.arange(0, total_time + time_step, time_step)

# Define collapse operators (for the Lindblad equation)
custom_matrix = Qobj([[0, 1], [0, 0]])
jump_operator = []
si = qeye(nmax + 1) 
for n in range(N):
  op_list = [si] * N  # Create a list of identity operators
  op_list[n] = custom_matrix  # Replace the n-th site with the custom matrix
  custom_op = tensor(op_list)  # Create the tensor product
  jump_operator.append(custom_op)

c_ops = [np.sqrt(gamma_relax) * a for a in a_list]


# Observables to calculate - total particle number
N_total = sum(n_list)

# Solve the Schrödinger equation (more efficient for this case)
result = mesolve(H, psi0, tlist, c_ops, [N_total])

# Extract expectation values
total_number = result.expect[0]

# Print results
print("Time evolution of total particle number:")
for t, n in zip(tlist, total_number):
    print(f"Time: {t:.2f}, Total number: {n:.4f}")

# Define SecondOrderOneSiteTDVP
gamma = 1

In [14]:
ket , bra = ptn.devectorize_pho(vectorized_pho , connections)

t = 0.4
U = 0.8
m = 0.4
creation_op, annihilation_op, number_op = ptn.bosonic_operators(d)
L = annihilation_op 
gamma = 1

# TTNO : Number operator
N = Number_op_total(3, 3, d)
N = N.pad_with_identities(ket, symbolic= True)
N = ptn.TTNO.from_hamiltonian(N, ket)

# TTNO : Liouville operator 
H1 = Liouville(t, U, gamma, m ,L, 3, 3, d)
H1 = H1.pad_with_identities(vectorized_pho , symbolic= True)
L_fancy = ptn.TTNO.from_hamiltonian(H1, vectorized_pho)

connections = [ ((0, 0), (0, 1), 1, 0),
                ((0, 1), (0, 2), 1, 0),
                ((0, 2), (1, 2), 1, 0),
                ((0, 0), (1, 0), 2, 0),
                ((1, 0), (2, 0), 1, 0),
                ((2, 0), (2, 1), 1, 0),
                ((2, 1), (1, 1), 1, 0),
                ((2, 1), (2, 2), 2, 0)]

tdvp_Lindblad = ptn.SecondOrderOneSiteTDVP(initial_state = vectorized_pho,
                                            hamiltonian = L_fancy,
                                            time_step_size = 0.01,
                                            final_time = 4,
                                            operators = N,
                                            connections = connections)

In [15]:
tdvp_Lindblad.run_Lindblad(evaluation_time=2)
times = tdvp_Lindblad.times

# Compare results 

In [16]:
import matplotlib.pyplot as plt

fig1, axs1 = plt.subplots(1, 1, sharex=True, figsize=(5, 5))


axs1.plot(total_number, label="exact" )
axs1.plot( tdvp_Lindblad.operator_results()[0] , label="tdvp", linestyle='--')




# axs1.plot(results_N_ttn, label="N_ttn")
# axs1.plot(results_N_t3n, label="N_t3n")

axs1.set_xlabel("Time $t$")
axs1.set_ylabel("Total Occupation ")
axs1.grid(True)
axs1.legend()